# Planning evaluation (Fig. 5)

Mean goal-conditioned planning success across seeds/repeats, one panel per environment
(paper Fig. 5, performance panels only). The figure is displayed inline.

In [ ]:
import sys
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def _repo_root(start):
    for c in [start.resolve(), *start.resolve().parents]:
        if (c / "pyproject.toml").exists() and (c / "experiments").is_dir():
            return c
    return start.resolve()


REPO_ROOT = _repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from plot_style import FULL_WIDTH, apply_matplotlib_style, palette

apply_matplotlib_style()

CSV_PATH = REPO_ROOT / "experiments" / "planning_eval" / "aggregated_results.csv"
print(CSV_PATH)

In [ ]:
ENV_ORDER = ["TwoRoom", "Reacher", "Push-T", "OGBench-Cube"]
METHOD_ORDER = ["Our", "SIGReg", "Random", "Forward only"]
ENV_LABELS = {"tworoom": "TwoRoom", "reacher": "Reacher", "pusht": "Push-T", "ogbcube": "OGBench-Cube"}
METHOD_LABELS = {"inverse": "Our", "sigreg": "SIGReg", "random": "Random", "forward_only": "Forward only"}

df = pd.read_csv(CSV_PATH)
df = df[df["status"].eq("ok")].copy()
df["success_rate"] = pd.to_numeric(df["success_rate"], errors="coerce")
df["environment"] = df["env"].map(ENV_LABELS)
df["method_label"] = df["method"].map(METHOD_LABELS)
df = df.dropna(subset=["success_rate", "environment", "method_label"])
if df.empty:
    raise ValueError("No completed planning-eval rows. Run aggregate_results.py after jobs finish.")

summary = (
    df.groupby(["environment", "method_label"])["success_rate"]
    .agg(mean="mean", std="std", n="count")
    .reset_index()
)
display(summary)

success_rates = {env: {} for env in ENV_ORDER}
success_errors = {env: {} for env in ENV_ORDER}
for _, r in summary.iterrows():
    success_rates[r["environment"]][r["method_label"]] = r["mean"]
    success_errors[r["environment"]][r["method_label"]] = r["std"]

In [ ]:
BAR_COLORS = {
    "Our": {"fill": palette["Light Red"], "edge": palette["Dark Red"]},
    "SIGReg": {"fill": palette["Light Blue"], "edge": palette["Dark Blue"]},
    "Random": {"fill": palette["Light Grey"], "edge": palette["Dark Grey"]},
    "Forward only": {"fill": palette["Light Purple"], "edge": palette["Dark Purple"]},
}


def lighten(color, amount=0.18):
    rgb = np.array(mcolors.to_rgb(color))
    return tuple(rgb + (1.0 - rgb) * amount)


def plot_success_bars(ax, env_name):
    x = np.arange(len(METHOD_ORDER))
    values = [success_rates[env_name].get(m, np.nan) for m in METHOD_ORDER]
    errors = [success_errors[env_name].get(m, np.nan) for m in METHOD_ORDER]
    edges = [BAR_COLORS[m]["edge"] for m in METHOD_ORDER]
    fills = [lighten(BAR_COLORS[m]["fill"]) for m in METHOD_ORDER]

    bars = ax.bar(
        x, values, yerr=errors, width=0.68, color=fills, edgecolor=edges,
        linewidth=0.8, capsize=2.5,
        error_kw={"elinewidth": 0.7, "capthick": 0.7, "ecolor": palette["Dark Grey"]},
    )
    for bar, value, edge in zip(bars, values, edges):
        if not np.isfinite(value):
            continue
        if value >= 18:
            y, va = value / 2.0, "center"
        else:
            y, va = value + 5.0, "bottom"
        ax.text(
            bar.get_x() + bar.get_width() / 2, y, f"{value:.0f}",
            ha="center", va=va, color=edge, fontsize=7, fontweight="bold", clip_on=False,
        )

    ax.set_ylim(0, 100)
    ax.set_yticks([0, 50, 100])
    ax.set_xticks(x)
    ax.set_xticklabels(METHOD_ORDER, rotation=38, ha="right", rotation_mode="anchor")
    ax.grid(axis="y", linewidth=0.45)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.margins(x=0.03)


fig, axes = plt.subplots(1, len(ENV_ORDER), figsize=(FULL_WIDTH, 2.0), constrained_layout=True)
for col, env_name in enumerate(ENV_ORDER):
    plot_success_bars(axes[col], env_name)
    axes[col].set_title(env_name, pad=4)
    if col == 0:
        axes[col].set_ylabel("Success rate (%)")
    else:
        axes[col].set_yticklabels([])
        axes[col].tick_params(axis="y", length=0)
plt.show()